# ATE for `visit` and `conversion`

Since randomization held (see notebook 01), the difference in average outcome between `treatment=1` and `treatment=0` is a valid estimate of the average treatment effect (ATE) — no adjustment needed.

`visit` and `conversion` are treated as two separate outcomes, each with its own ATE — not a funnel.

In [1]:
import sys
sys.path.insert(0, "../src")

from uplift.data import load_sample
from uplift.ate import compute_ate

df = load_sample()
df.shape

(1000000, 16)

## Difference-in-means ATE with 95% CI

For a 0/1 outcome, each group's variance is `p(1-p)`. Treatment and control are independent samples, so their variances add:

```
ATE = p_treated - p_control
SE  = sqrt(p_treated*(1-p_treated)/n_treated + p_control*(1-p_control)/n_control)
95% CI = ATE +/- 1.96 * SE
```

In [2]:
results = {outcome: compute_ate(df, outcome) for outcome in ["visit", "conversion"]}

for outcome, r in results.items():
    print(f"--- {outcome} ---")
    print(f"n_treated={r.n_treated}, n_control={r.n_control}")
    print(f"rate_treated={r.rate_treated:.5f}, rate_control={r.rate_control:.5f}")
    print(f"ATE={r.ate:.5f}  (95% CI: {r.ci_low:.5f} to {r.ci_high:.5f})")
    print(f"relative lift={r.relative_lift:.2%}")
    print()

--- visit ---
n_treated=850849, n_control=149151
rate_treated=0.04860, rate_control=0.03779
ATE=0.01081  (95% CI: 0.00974 to 0.01188)
relative lift=28.61%

--- conversion ---
n_treated=850849, n_control=149151
rate_treated=0.00312, rate_control=0.00201
ATE=0.00111  (95% CI: 0.00085 to 0.00137)
relative lift=55.25%



## Verify: relative CI width, visit vs. conversion

Even though `conversion`'s ATE and absolute CI are numerically smaller, is its CI *proportionally* wider than `visit`'s, confirming that the rarer outcome is noisier relative to its own effect size?

In [3]:
for outcome, r in results.items():
    rel_half_width = (r.ci_high - r.ate) / r.ate
    print(f"{outcome}: relative half-width = {rel_half_width:.2%}")

visit: relative half-width = 9.90%
conversion: relative half-width = 23.07%


## Retrospective power / minimum detectable effect (MDE)

The 95% CI tells you whether *this observed* result is statistically significant. It doesn't tell you whether the experiment's design was sensitive enough to reliably catch a real effect in the first place.

MDE answers a different question: if the true effect were exactly at some size, what fraction of repeated experiments (at this sample size, this base rate) would successfully produce a significant result? We target 80% power (catch a real effect 4 times out of 5).

Derivation: a CI's significance cutoff is `1.96 * SE` (for a two-sided 95% test) — at that cutoff, the true effect and the cutoff coincide, so by symmetry of the sampling distribution only 50% of repeated experiments would land above it (50% power, a coin flip). To reach 80% power, the true effect needs to sit further out, by an extra `z_power * SE` (`z_power ≈ 0.84` for 80% power):

```
MDE = (z_alpha/2 + z_power) * SE ≈ (1.96 + 0.84) * SE ≈ 2.80 * SE
```

roughly 1.4x the raw significance margin.

In [ ]:
from uplift.ate import compute_mde

for outcome, r in results.items():
    m = compute_mde(r)
    print(f"--- {outcome} ---")
    print(f"observed ATE={r.ate:.5f} (relative lift {r.relative_lift:.2%})")
    print(f"MDE (80% power, alpha=0.05): abs={m.mde_abs:.5f}, relative={m.mde_relative:.2%}")
    print()

## Conclusion

Both outcomes' observed effects sit well above their own MDE — `visit` at 28.6% relative lift vs. an MDE of ~4.1%, `conversion` at 55.3% vs. an MDE of ~18.2%. `conversion`'s MDE is ~4.5x larger than `visit`'s, confirming it's the noisier outcome (far fewer events at a 0.29% base rate).

Practical takeaway: this experiment was well-powered for both outcomes. Neither result is a fragile, borderline-significant finding — both comfortably clear the bar of what this design could reliably detect, which is a stronger claim than "the CI excludes zero" on its own.